# اليوم الثاني — مختبر 3A: تصنيف النصوص
## Day 2 — Lab 3A: Text Classification

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار:** Core → Explore → Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manal-qaysi/bayan-nlp-Manal-qaysi/blob/main/notebooks/03_text_classification.ipynb)

> نبني Baseline أولًا، نثبت عدم تسرب المجموعات، ثم ننفذ خطوة ضبط فعلية لـDistilmBERT متعدد اللغات.
>
> Build a baseline, prove group isolation, then perform a real fine-tuning step on multilingual DistilBERT.

**علامة النجاح:** `DAY2_NOTEBOOK3_CORE=PASS`.

> 🟣 **نسخة مشروع منال قيسي (Manal-qaysi).** المختبر الأصلي من مادة الدورة، والخلايا المعنونة بـ«🟣 مشروعي» أُضيفت لمشروعي: تربط الدفتر بمستودعي وبياناته وتطبع نتائج قابلة للتتبع بصيغة `BAYAN_RESULT::…` يجمعها `scripts/collect_results.py`.
> Run with **Runtime → Restart session and run all**, then **File → Save a copy in GitHub** to `notebooks/03_text_classification.ipynb` in `Manal-qaysi/bayan-nlp-Manal-qaysi`.

## قواعد المختبر

- البيانات اصطناعية ولا تحتوي حالات أو أشخاصًا حقيقيين.
- GPU في Colab Free غير مضمون؛ CPU fallback يجمد المشفر ويدرب task head.
- لا نرفع model weights أو cache إلى GitHub.
- نتائج هذه العينة الصغيرة تسمى `MEASURED_SMOKE`، وليست أداءً إنتاجيًا.
- نفذ **Runtime → Run all**. لا تتجاوز خلية فاشلة.

In [ ]:
# تثبيت نسخ اليوم الثاني فقط عند الحاجة
import importlib.metadata
import importlib.util
import subprocess
import sys

REQUIRED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "scikit-learn": "1.9.0",
}
needs_install = []
for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != expected:
        needs_install.append(f"{distribution}=={expected}")

if needs_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needs_install])

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is required. Open this notebook in Google Colab.")

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

In [ ]:
# 🟣 مشروعي — ربط الدفتر بمستودعي: نسخة الكود والبيانات وبصماتها (SHA-256)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Manal-qaysi/bayan-nlp-Manal-qaysi"
_candidates = [Path.cwd(), Path.cwd().parent, Path("/content/bayan-nlp-Manal-qaysi")]
REPO_DIR = next((p for p in _candidates if (p / "src" / "bayan" / "project.py").is_file()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/bayan-nlp-Manal-qaysi") if Path("/content").is_dir() else Path("bayan-nlp-Manal-qaysi").resolve()
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from bayan.project import data_hashes, emit_result, git_head

DATA_DIR = REPO_DIR / "data" / "sample"
REPO_SHA = git_head(REPO_DIR)
DATA_SHA256 = data_hashes(DATA_DIR)
PROVENANCE = {"repo": REPO_URL, "repo_sha": REPO_SHA}
print("REPO_DIR:", REPO_DIR)
print("REPO_SHA:", REPO_SHA)
for _name, _digest in DATA_SHA256.items():
    print(f"  {_name}  sha256={_digest[:16]}…")

In [ ]:
import csv
import io
import json
import math
import os
import random
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1) تحميل البيانات وفحصها

يحاول الدفتر قراءة ملف الدورة من GitHub. إذا تعذر، يستخدم عينة اصطناعية مدمجة حتى لا يتوقف Core بسبب رابط البيانات. النموذج نفسه يحتاج تنزيلًا أول مرة.

In [ ]:
DATA_URL = (DATA_DIR / "bayan_day2_classification.csv").as_uri()
FALLBACK_ROWS = [{"example_id":"F-001","group_id":"DG-A","split":"train","language":"ar","text":"تعذر تسجيل الدخول إلى البوابة","topic":"digital_service","sentiment":"negative"},{"example_id":"F-002","group_id":"DG-A","split":"train","language":"en","text":"I cannot sign in to the portal","topic":"digital_service","sentiment":"negative"},{"example_id":"F-003","group_id":"DG-B","split":"train","language":"ar","text":"الخدمة الإلكترونية سريعة وواضحة","topic":"digital_service","sentiment":"positive"},{"example_id":"F-004","group_id":"DG-B","split":"train","language":"en","text":"The online service is clear and fast","topic":"digital_service","sentiment":"positive"},{"example_id":"F-005","group_id":"PG-A","split":"train","language":"ar","text":"أحتاج معرفة حالة طلب التصريح","topic":"permit","sentiment":"neutral"},{"example_id":"F-006","group_id":"PG-A","split":"train","language":"en","text":"I need the status of my permit request","topic":"permit","sentiment":"neutral"},{"example_id":"F-007","group_id":"PG-B","split":"train","language":"ar","text":"تمت الموافقة على التصريح اليوم","topic":"permit","sentiment":"positive"},{"example_id":"F-008","group_id":"PG-B","split":"train","language":"en","text":"The permit was approved today","topic":"permit","sentiment":"positive"},{"example_id":"F-009","group_id":"HG-A","split":"train","language":"ar","text":"تأخر موعد العيادة هذا الصباح","topic":"health","sentiment":"negative"},{"example_id":"F-010","group_id":"HG-A","split":"train","language":"en","text":"My clinic appointment was delayed","topic":"health","sentiment":"negative"},{"example_id":"F-011","group_id":"HG-B","split":"train","language":"ar","text":"كانت خدمة العيادة ممتازة","topic":"health","sentiment":"positive"},{"example_id":"F-012","group_id":"HG-B","split":"train","language":"en","text":"The clinic service was excellent","topic":"health","sentiment":"positive"},{"example_id":"F-013","group_id":"TG-A","split":"train","language":"ar","text":"الحافلة لم تصل في الوقت المحدد","topic":"transport","sentiment":"negative"},{"example_id":"F-014","group_id":"TG-A","split":"train","language":"en","text":"The bus did not arrive on time","topic":"transport","sentiment":"negative"},{"example_id":"F-015","group_id":"TG-B","split":"train","language":"ar","text":"كانت الرحلة مريحة ومنظمة","topic":"transport","sentiment":"positive"},{"example_id":"F-016","group_id":"TG-B","split":"train","language":"en","text":"The trip was comfortable and organised","topic":"transport","sentiment":"positive"},{"example_id":"F-017","group_id":"DG-V","split":"validation","language":"ar","text":"لم يصل رمز التحقق الرقمي","topic":"digital_service","sentiment":"negative"},{"example_id":"F-018","group_id":"PG-V","split":"validation","language":"en","text":"How can I renew the permit","topic":"permit","sentiment":"neutral"},{"example_id":"F-019","group_id":"HG-V","split":"validation","language":"ar","text":"أحتاج إعادة جدولة الموعد الصحي","topic":"health","sentiment":"neutral"},{"example_id":"F-020","group_id":"TG-V","split":"validation","language":"en","text":"The bus route has changed","topic":"transport","sentiment":"neutral"},{"example_id":"F-021","group_id":"DG-T","split":"test","language":"en","text":"The verification code did not arrive","topic":"digital_service","sentiment":"negative"},{"example_id":"F-022","group_id":"PG-T","split":"test","language":"ar","text":"تأخر إصدار التصريح المطلوب","topic":"permit","sentiment":"negative"},{"example_id":"F-023","group_id":"HG-T","split":"test","language":"en","text":"The clinic appointment was cancelled","topic":"health","sentiment":"negative"},{"example_id":"F-024","group_id":"TG-T","split":"test","language":"ar","text":"توقفت الحافلة قبل المحطة","topic":"transport","sentiment":"negative"}]

try:
    with urllib.request.urlopen(DATA_URL, timeout=20) as response:
        text = response.read().decode("utf-8")
    rows = list(csv.DictReader(io.StringIO(text)))
    DATA_SOURCE = "project_repo_file"
except Exception as exc:
    rows = FALLBACK_ROWS
    DATA_SOURCE = f"embedded_fallback:{type(exc).__name__}"

print("Data source:", DATA_SOURCE)
print("Rows:", len(rows))
print("Topics:", Counter(row["topic"] for row in rows))
assert len(rows) >= 24
assert {"ar", "en"} <= {row["language"] for row in rows}

In [ ]:
# 🟣 مشروعي — عقد النسختين: display_text للعرض، و text = نسخة النموذج بعقد المعالجة الموحّد
from bayan.project import PREPROCESSING_VERSION, prepare_model_text

for row in rows:
    row["display_text"] = row["text"]
    row["text"] = prepare_model_text(row["display_text"], row["language"])
print(PREPROCESSING_VERSION, "applied to", len(rows), "rows; changed:",
      sum(row["text"] != row["display_text"] for row in rows))
print("sentiment by split:", {split: dict(Counter(r["sentiment"] for r in rows if r["split"] == split))
                              for split in ["train", "validation", "test"]})

### عقد رأسي topic وsentiment / Two independent classification heads

يحتوي ملف البيانات على `topic` و`sentiment`. تشغيل المرجع المحفوظ يقيس رأس **topic** كي يبقى smoke محدودًا، لكنه لا يُعد دليلًا لرأس sentiment. في مشروع بيان أعد استخدام split المجمّد نفسه لتدريب رأس sentiment مستقل وlabel map مستقلة، ثم قيّم الرأسين كلًا بمقياسه. تحافظ نسخة fallback كذلك على العمودين حتى لا يتغير العقد عند تعطل GitHub Raw.

In [ ]:
def validate_splits(rows):
    required = {"train", "validation", "test"}
    group_owner = {}
    labels_by_split = {name: set() for name in required}
    counts = Counter()
    for row in rows:
        split, group, label = row["split"], row["group_id"], row["topic"]
        if split not in required:
            raise ValueError(f"Unknown split: {split}")
        previous = group_owner.setdefault(group, split)
        if previous != split:
            raise ValueError(f"Group leakage: {group}")
        labels_by_split[split].add(label)
        counts[split] += 1
    all_labels = set().union(*labels_by_split.values())
    for split in required:
        if labels_by_split[split] != all_labels:
            raise ValueError(f"Missing label in {split}")
    return {
        "rows": dict(counts),
        "groups": len(group_owner),
        "labels": sorted(all_labels),
        "group_overlap": 0,
    }

split_report = validate_splits(rows)
print(json.dumps(split_report, ensure_ascii=False, indent=2))
assert split_report["group_overlap"] == 0
print("Split contract=PASS")

**المتوقع:** كل split يحتوي الفئات الأربع، و`group_overlap` يساوي صفرًا.

## 2) TF-IDF Baseline

نضبط الـbaseline على Train ونقرأ Validation. لا نستخدم Test حتى خلية القياس النهائي بعد تثبيت المسار.

In [ ]:
train_rows = [row for row in rows if row["split"] == "train"]
validation_rows = [row for row in rows if row["split"] == "validation"]
test_rows = [row for row in rows if row["split"] == "test"]
LABELS = sorted({row["topic"] for row in rows})
label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1),
    LinearSVC(random_state=SEED),
)
baseline.fit(
    [row["text"] for row in train_rows],
    [row["topic"] for row in train_rows],
)
baseline_val_pred = baseline.predict([row["text"] for row in validation_rows])
baseline_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    baseline_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
print("Baseline validation macro-F1 (MEASURED_SMOKE):", round(baseline_val_f1, 4))
assert 0.0 <= baseline_val_f1 <= 1.0
print("Baseline=PASS")

## 3) تنزيل الـcheckpoint وتجهيز النموذج

النموذج عام ومجاني، ولا يحتاج Hugging Face token. عند ظهور warning بأن classification head جديدة فهذا متوقع: المشفر مدرب مسبقًا، أما رأس فئات بيان فيبدأ عشوائيًا.

- GPU: Full fine-tuning قصير.
- CPU: Partial fine-tuning؛ نجمّد معظم المشفر ونحدّث آخر Transformer block مع رأس المهمة.

كلاهما تدريب فعلي، لكن نوعه يسجل في النتائج.

In [ ]:
MODEL_ID = "distilbert/distilbert-base-multilingual-cased"
MAX_LENGTH = 64
BATCH_SIZE = 4
NUM_EPOCHS = 2 if DEVICE.type == "cuda" else 12

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    )
except Exception as exc:
    raise RuntimeError(
        "Checkpoint download failed. Reconnect the runtime and run this cell once. "
        "No API key is required."
    ) from exc

TRAINING_MODE = "full_finetune" if DEVICE.type == "cuda" else "partial_finetune_cpu"
if TRAINING_MODE == "partial_finetune_cpu":
    for parameter in model.base_model.parameters():
        parameter.requires_grad = False
    for parameter in model.base_model.transformer.layer[-1].parameters():
        parameter.requires_grad = True

model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Training mode:", TRAINING_MODE)
print("Epochs:", NUM_EPOCHS)
print(f"Trainable parameters: {trainable:,} / {total:,}")
assert trainable > 0

In [ ]:
# 🟣 مشروعي — اختيار MAX_LENGTH من أطوال الرموز الفعلية (train+validation) بدل 64 الثابتة — D-002
from bayan.project import choose_max_length, length_profile

CANDIDATE_LENGTHS = [16, 24, 32, 48, 64, 96, 128]
_length_rows = [row for row in rows if row["split"] in {"train", "validation"}]
_token_lengths = [len(tokenizer(row["text"])["input_ids"]) for row in _length_rows]
length_evidence = length_profile(_token_lengths, [len(row["text"].split()) for row in _length_rows], CANDIDATE_LENGTHS)
max_length_choice = choose_max_length(_token_lengths, CANDIDATE_LENGTHS)
MAX_LENGTH = max_length_choice["max_length"]
print("token length p50/p95/max:", length_evidence["p50_tokens"], length_evidence["p95_tokens"], length_evidence["max_tokens"])
print("truncation by candidate:", length_evidence["truncation_rate"])
print("MAX_LENGTH =", MAX_LENGTH, "|", max_length_choice["rule"])

In [ ]:
def iter_batches(examples, batch_size, *, shuffle=False, seed=SEED):
    indexes = np.arange(len(examples))
    if shuffle:
        np.random.default_rng(seed).shuffle(indexes)
    for start in range(0, len(indexes), batch_size):
        yield [examples[index] for index in indexes[start:start + batch_size]]


def encode_batch(batch):
    encoded = tokenizer(
        [row["text"] for row in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(
        [label2id[row["topic"]] for row in batch], dtype=torch.long
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}


def predict(examples):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in iter_batches(examples, BATCH_SIZE):
            encoded = encode_batch(batch)
            logits = model(**encoded).logits
            predictions.extend(logits.argmax(-1).cpu().tolist())
    return [id2label[index] for index in predictions]

print("Batch functions=PASS")

## 4) خطوة Fine-tuning فعلية

نحدّث أوزانًا داخل آخر Transformer block، ونستخدم validation لاختيار أفضل حقبة، ثم نفتح Test مرة واحدة بعد تثبيت الإعدادات. العينة صغيرة واصطناعية؛ لذا المقارنة دليل تعليمي على سلامة المسار وليست تقديرًا للإنتاج.

In [ ]:
learning_rate = 2e-5 if TRAINING_MODE == "full_finetune" else 1e-4
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_parameters, lr=learning_rate)
losses = []
train_steps = 0
epoch_history = []
best_validation_f1 = -1.0
best_epoch = 0
best_trainable_state = None

for epoch_index in range(NUM_EPOCHS):
    model.train()
    epoch_losses = []
    for batch in iter_batches(
        train_rows, BATCH_SIZE, shuffle=True, seed=SEED + epoch_index + 1
    ):
        optimizer.zero_grad(set_to_none=True)
        encoded = encode_batch(batch)
        output = model(**encoded)
        loss = output.loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
        optimizer.step()
        loss_value = float(loss.detach().cpu())
        losses.append(loss_value)
        epoch_losses.append(loss_value)
        train_steps += 1

    epoch_predictions = predict(validation_rows)
    epoch_f1 = f1_score(
        [row["topic"] for row in validation_rows], epoch_predictions,
        labels=LABELS, average="macro", zero_division=0,
    )
    epoch_history.append({
        "epoch": epoch_index + 1,
        "mean_loss": float(np.mean(epoch_losses)),
        "validation_macro_f1": float(epoch_f1),
    })
    if epoch_f1 > best_validation_f1:
        best_validation_f1 = float(epoch_f1)
        best_epoch = epoch_index + 1
        if TRAINING_MODE == "partial_finetune_cpu":
            best_trainable_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter in model.named_parameters()
                if parameter.requires_grad
            }
    print(
        f"epoch={epoch_index + 1:02d} mean_loss={np.mean(epoch_losses):.4f} "
        f"validation_macro_f1={epoch_f1:.4f}"
    )

if best_trainable_state is not None:
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in best_trainable_state:
                parameter.copy_(best_trainable_state[name].to(parameter.device))
    selected_epoch = best_epoch
else:
    selected_epoch = NUM_EPOCHS

assert train_steps >= 1 and all(math.isfinite(value) for value in losses)
assert any(parameter.requires_grad for parameter in model.base_model.parameters())
print("Transformer optimizer steps=PASS", {"steps": train_steps, "selected_epoch": selected_epoch})

In [ ]:
transformer_val_pred = predict(validation_rows)
transformer_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    transformer_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
validation_delta = transformer_val_f1 - baseline_val_f1
print("Selected epoch:", selected_epoch)
print("Transformer validation macro-F1 (MEASURED_SMOKE):", round(transformer_val_f1, 4))
print("Validation delta vs TF-IDF baseline:", round(validation_delta, 4))
print("Validation predictions:", list(zip(
    [row["topic"] for row in validation_rows], transformer_val_pred
)))
assert 0.0 <= transformer_val_f1 <= 1.0

## 5) القياس النهائي المصغر

بعد تثبيت الإعدادات السابقة نقيس Test مرة واحدة. لا نغير الإعدادات بناء على هذه النتيجة ثم نعيد تسميتها Frozen Test.

In [ ]:
test_truth = [row["topic"] for row in test_rows]
baseline_test_pred = baseline.predict([row["text"] for row in test_rows]).tolist()
transformer_test_pred = predict(test_rows)

results = {
    "result_type": "MEASURED_SMOKE",
    "data_source": DATA_SOURCE,
    "model_id": MODEL_ID,
    "device": str(DEVICE),
    "training_mode": TRAINING_MODE,
    "seed": SEED,
    "epochs_run": NUM_EPOCHS,
    "selected_epoch": selected_epoch,
    "train_steps": train_steps,
    "mean_train_loss": float(np.mean(losses)),
    "baseline_validation_macro_f1": float(baseline_val_f1),
    "transformer_validation_macro_f1": float(transformer_val_f1),
    "validation_delta_vs_baseline": float(validation_delta),
    "baseline_beaten_on_validation": bool(validation_delta > 0),
    "baseline_test_macro_f1": float(f1_score(
        test_truth, baseline_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_macro_f1": float(f1_score(
        test_truth, transformer_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_accuracy": float(accuracy_score(test_truth, transformer_test_pred)),
    "limitations": [
        "synthetic tiny dataset",
        "small validation set used for epoch selection",
        "not an estimate of production quality",
    ],
}
print(json.dumps(results, ensure_ascii=False, indent=2))
Path("day2_classification_metrics.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
)

## 🟣 مشروعي — رأس sentiment مستقل على التقسيم المجمّد نفسه (T3)

أدرّب رأسًا ثانيًا للمشاعر بـlabel map مستقلة، على نفس split والبذرة ونمط التدريب (full على GPU، وعلى CPU آخر طبقة مع الرأس)، ومعه baseline من TF-IDF. أختار الحقبة على validation، ثم أقرأ test مرة واحدة.

> **قيد في البيانات:** validation فيها `neutral/negative` فقط، وtest فيها `negative/positive` فقط. لذلك يُحسب Macro-F1 للمشاعر على الفئات **الموجودة فعلًا** في الحقيقة والتنبؤ، وأذكر هذا القيد في التقرير.

In [ ]:
# 🟣 مشروعي — تدريب رأس sentiment: baseline + Transformer
SENTIMENT_LABELS = sorted({row["sentiment"] for row in rows})
s_label2id = {label: index for index, label in enumerate(SENTIMENT_LABELS)}
s_id2label = {index: label for label, index in s_label2id.items()}

def observed_macro_f1(truth, prediction):
    labels = sorted(set(truth) | set(prediction))
    return float(f1_score(truth, prediction, labels=labels, average="macro", zero_division=0))

sentiment_baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1),
    LinearSVC(random_state=SEED),
)
sentiment_baseline.fit([r["text"] for r in train_rows], [r["sentiment"] for r in train_rows])

torch.manual_seed(SEED)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(SENTIMENT_LABELS), label2id=s_label2id, id2label=s_id2label,
)
if TRAINING_MODE == "partial_finetune_cpu":
    for parameter in sentiment_model.base_model.parameters():
        parameter.requires_grad = False
    for parameter in sentiment_model.base_model.transformer.layer[-1].parameters():
        parameter.requires_grad = True
sentiment_model.to(DEVICE)

def encode_sentiment(batch):
    encoded = tokenizer([r["text"] for r in batch], padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt")
    encoded["labels"] = torch.tensor([s_label2id[r["sentiment"]] for r in batch], dtype=torch.long)
    return {key: value.to(DEVICE) for key, value in encoded.items()}

def predict_sentiment(examples):
    sentiment_model.eval()
    output = []
    with torch.no_grad():
        for batch in iter_batches(examples, BATCH_SIZE):
            output.extend(sentiment_model(**encode_sentiment(batch)).logits.argmax(-1).cpu().tolist())
    return [s_id2label[index] for index in output]

s_parameters = [p for p in sentiment_model.parameters() if p.requires_grad]
s_optimizer = AdamW(s_parameters, lr=learning_rate)
s_best_f1, s_best_epoch, s_best_state, s_steps, s_losses = -1.0, 0, None, 0, []
for epoch_index in range(NUM_EPOCHS):
    sentiment_model.train()
    for batch in iter_batches(train_rows, BATCH_SIZE, shuffle=True, seed=SEED + 100 + epoch_index):
        s_optimizer.zero_grad(set_to_none=True)
        loss = sentiment_model(**encode_sentiment(batch)).loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite sentiment loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(s_parameters, 1.0)
        s_optimizer.step()
        s_losses.append(float(loss.detach().cpu()))
        s_steps += 1
    s_val_f1 = observed_macro_f1([r["sentiment"] for r in validation_rows], predict_sentiment(validation_rows))
    if s_val_f1 > s_best_f1:
        s_best_f1, s_best_epoch = s_val_f1, epoch_index + 1
        s_best_state = {n: p.detach().cpu().clone() for n, p in sentiment_model.named_parameters() if p.requires_grad}
    print(f"sentiment epoch={epoch_index + 1:02d} validation_macro_f1={s_val_f1:.4f}")

with torch.no_grad():
    for name, parameter in sentiment_model.named_parameters():
        if name in s_best_state:
            parameter.copy_(s_best_state[name].to(parameter.device))
assert s_steps >= 1 and all(math.isfinite(v) for v in s_losses)
print("Sentiment head trained", {"steps": s_steps, "selected_epoch": s_best_epoch, "validation_macro_f1": round(s_best_f1, 4)})

In [ ]:
# 🟣 مشروعي — تقييم الرأسين (validation ثم test المجمّد) وحفظ كل تنبؤ للتتبع في دفتر 07
def prediction_table(examples, split_name):
    texts = [r["text"] for r in examples]
    return [
        {
            "example_id": r["example_id"], "split": split_name, "language": r["language"],
            "text": r["display_text"],
            "topic_gold": r["topic"], "topic_baseline": b_topic, "topic_transformer": t_topic,
            "sentiment_gold": r["sentiment"], "sentiment_baseline": b_sent, "sentiment_transformer": t_sent,
        }
        for r, b_topic, t_topic, b_sent, t_sent in zip(
            examples,
            baseline.predict(texts).tolist(), predict(examples),
            sentiment_baseline.predict(texts).tolist(), predict_sentiment(examples),
        )
    ]

prediction_rows = prediction_table(validation_rows, "validation") + prediction_table(test_rows, "test")

def head_metrics(split_name, head):
    subset = [r for r in prediction_rows if r["split"] == split_name]
    truth = [r[f"{head}_gold"] for r in subset]
    if head == "topic":
        metric = lambda t, p: float(f1_score(t, p, labels=LABELS, average="macro", zero_division=0))
    else:
        metric = observed_macro_f1
    return {
        "n": len(subset),
        "labels_in_truth": sorted(set(truth)),
        "baseline_macro_f1": metric(truth, [r[f"{head}_baseline"] for r in subset]),
        "transformer_macro_f1": metric(truth, [r[f"{head}_transformer"] for r in subset]),
        "baseline_accuracy": float(accuracy_score(truth, [r[f"{head}_baseline"] for r in subset])),
        "transformer_accuracy": float(accuracy_score(truth, [r[f"{head}_transformer"] for r in subset])),
    }

head_results = {head: {split: head_metrics(split, head) for split in ["validation", "test"]}
                for head in ["topic", "sentiment"]}
print(json.dumps(head_results, ensure_ascii=False, indent=2))
if abs(head_results["topic"]["test"]["transformer_macro_f1"] - results["transformer_test_macro_f1"]) > 1e-9:
    print("WARNING: topic test F1 differs from the value computed above; check for non-deterministic inference")
for r in prediction_rows:
    wrong = r["topic_transformer"] != r["topic_gold"] or r["sentiment_transformer"] != r["sentiment_gold"]
    print(r["split"], r["example_id"], r["language"], "| topic", r["topic_gold"], "→", r["topic_transformer"],
          "| sentiment", r["sentiment_gold"], "→", r["sentiment_transformer"], "  ← error" if wrong else "")

In [ ]:
# 🟣 مشروعي — اختبارات سلوكية (invariance): تغييرات شكلية لا يجب أن تغيّر الموضوع
from bayan.error_analysis import behavioural_pass_rate

def perturbations(text, language):
    variants = {"trailing_punctuation": text + " !", "double_spaces": text.replace(" ", "  ")}
    if language == "ar":
        variants["tatweel"] = text.replace("ا", "ـا", 1)
        variants["diacritic"] = text.replace("ل", "لَ", 1)
    else:
        variants["lowercase"] = text.lower()
    return variants

behaviour_rows = []
for r in validation_rows + test_rows:
    original = predict([r])[0]
    for name, changed_text in perturbations(r["display_text"], r["language"]).items():
        changed_row = {**r, "text": prepare_model_text(changed_text, r["language"])}
        behaviour_rows.append({
            "example_id": r["example_id"], "perturbation": name, "language": r["language"],
            "passed": predict([changed_row])[0] == original,
        })
behaviour_summary = behavioural_pass_rate(behaviour_rows)
behaviour_by_type = {
    name: behavioural_pass_rate([b for b in behaviour_rows if b["perturbation"] == name])
    for name in sorted({b["perturbation"] for b in behaviour_rows})
}
print("INVARIANCE", behaviour_summary)
for name, summary in behaviour_by_type.items():
    print("  ", name, summary)

In [ ]:
# 🟣 مشروعي — حفظ نموذج topic وملف validation في Drive لدفتر 08 (خارج GitHub؛ لا تُرفع الأوزان)
IN_COLAB = Path("/content").is_dir()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PRIVATE_ARTIFACT_DIR = Path("/content/drive/MyDrive/bayan/model-v1")
else:
    PRIVATE_ARTIFACT_DIR = REPO_DIR / "artifacts" / "model-v1"   # ignored by .gitignore
PRIVATE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(PRIVATE_ARTIFACT_DIR)
tokenizer.save_pretrained(PRIVATE_ARTIFACT_DIR)
with (PRIVATE_ARTIFACT_DIR / "validation.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["example_id", "split", "language", "text", "label"])
    writer.writeheader()
    for r in validation_rows:
        writer.writerow({"example_id": r["example_id"], "split": "validation", "language": r["language"],
                         "text": r["display_text"], "label": r["topic"]})
training_meta = {
    "model_id": MODEL_ID, "training_mode": TRAINING_MODE, "max_length": MAX_LENGTH,
    "preprocessing_version": PREPROCESSING_VERSION, "labels": LABELS, "repo_sha": REPO_SHA,
    "selected_epoch": selected_epoch,
}
(PRIVATE_ARTIFACT_DIR / "bayan_training_meta.json").write_text(
    json.dumps(training_meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved private artefact for notebook 08 (not committed):", sorted(p.name for p in PRIVATE_ARTIFACT_DIR.iterdir()))

In [ ]:
# 🟣 مشروعي — حفظ الدليل القابل للتتبع (يقرؤه دفتر 07 و scripts/collect_results.py)
emit_result("nb03_classification", {
    **PROVENANCE,
    "data_sha256": DATA_SHA256.get("bayan_day2_classification.csv"),
    "preprocessing_version": PREPROCESSING_VERSION,
    "split_report": split_report,
    "max_length_choice": max_length_choice,
    "length_evidence": length_evidence,
    "topic_run": results,
    "heads": head_results,
    "sentiment_training": {"steps": s_steps, "selected_epoch": s_best_epoch,
                           "validation_macro_f1": s_best_f1, "labels": SENTIMENT_LABELS},
    "behavioural": {"overall": behaviour_summary, "by_perturbation": behaviour_by_type},
    "training_meta": training_meta,
    "predictions": prediction_rows,
})

لا تستنتج أن Transformer «فشل» أو «نجح إنتاجيًا» من هذه العينة. سجل الفرق، ثم اكتب ما تحتاجه لتقدير موثوق: بيانات أكثر، عدة بذور، slices، وتقييم مجمد.

## مستويات التحدي

- **Core:** الخلايا السابقة فقط.
- **Explore:** على GPU قارن full fine-tuning بالمسار الجزئي، مع تثبيت البيانات والبذرة والـvalidation.
- **Distinction:** نفذ ثلاث بذور وسجل mean ± range؛ لا تستخدم Test لاختيار البذرة.

In [ ]:
core_checks = {
    "split_isolation": split_report["group_overlap"] == 0,
    "baseline_valid": 0.0 <= baseline_val_f1 <= 1.0,
    "training_ran": train_steps >= 1,
    "transformer_weights_updated": any(
        parameter.requires_grad for parameter in model.base_model.parameters()
    ),
    "loss_is_finite": all(math.isfinite(value) for value in losses),
    "validation_valid": 0.0 <= transformer_val_f1 <= 1.0,
    "baseline_comparison_recorded": math.isfinite(validation_delta),
    "result_is_honest": results["result_type"] == "MEASURED_SMOKE",
    "bilingual_data": {"ar", "en"} <= {row["language"] for row in rows},
    "sentiment_head_trained": s_steps >= 1,
    "max_length_measured": MAX_LENGTH in CANDIDATE_LENGTHS,
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY2_NOTEBOOK3_CORE=PASS")

## نقطة GitHub

1. احفظ نسخة notebook في مستودعك بالاسم نفسه.
2. لا ترفع Hugging Face cache أو model weights.
3. ارفع `day2_classification_metrics.json` ضمن `reports/` إذا لم يحتو بيانات شخصية.
4. حدّث `DECISIONS.md` بنوع التدريب والسبب وحدود العينة.
5. لا تعمل commit النهائي لليوم حتى تكمل دفتر NER وQA.

**التالي:** [مختبر NER وQA](04_ner_and_qa.ipynb).